In [1]:
import pandas as pd
import numpy as np

# ============================================================================
# LOAD BOTH DATASETS
# ============================================================================

print("Loading datasets...")

# Dataset 1: Your predictions dataset
df1 = pd.read_parquet("../Data/Parquet/daylight_satellite_all_predictions.parquet")
print(f"Dataset 1 (predictions): {len(df1)} rows")

# Dataset 2: Your photic zone dataset with actual Secchi
df2 = pd.read_parquet("../Data/Parquet/New_estchlsummed.parquet")  # UPDATE PATH
print(f"Dataset 2 (actual Secchi): {len(df2)} rows")

# ============================================================================
# EXTRACT DATE INFORMATION
# ============================================================================

# Dataset 1
df1['Date'] = pd.to_datetime(df1['DATE_TIME_UTC'])
df1['Month'] = df1['Date'].dt.month
df1['Year'] = df1['Date'].dt.year

# Dataset 2
df2['Date'] = pd.to_datetime(df2['DATE_TIME_UTC'])
df2['Month'] = df2['Date'].dt.month
df2['Year'] = df2['Date'].dt.year

# ============================================================================
# FILTER TO SPRING MONTHS (March, April, May)
# ============================================================================

spring_months = [3, 4, 5, 6, 7, 8]

df1_spring = df1[df1['Month'].isin(spring_months)]
df2_spring = df2[df2['Month'].isin(spring_months)]

print(f"\nSpring data:")
print(f"  Dataset 1: {len(df1_spring)} rows")
print(f"  Dataset 2: {len(df2_spring)} rows")

# ============================================================================
# COUNT STATIONS PER CRUISE IN EACH DATASET
# ============================================================================

print("\n" + "="*80)
print("COUNTING STATIONS PER CRUISE")
print("="*80)

# Count in dataset 1
cruise_counts_1 = df1_spring.groupby('Cruise_ID').agg({
    'CAST_COUNT': 'count',
    'Year': 'first',
    'Month': 'first'
}).rename(columns={'CAST_COUNT': 'N_Stations_Dataset1'})

# Count in dataset 2
cruise_counts_2 = df2_spring.groupby('Cruise_ID').agg({
    'CAST_COUNT': 'count',
    'Year': 'first',
    'Month': 'first'
}).rename(columns={'CAST_COUNT': 'N_Stations_Dataset2'})

# Merge counts
cruise_comparison = cruise_counts_1.join(cruise_counts_2, how='outer', rsuffix='_2')

# Fill NaN with 0 (cruise exists in one dataset but not the other)
cruise_comparison['N_Stations_Dataset1'] = cruise_comparison['N_Stations_Dataset1'].fillna(0).astype(int)
cruise_comparison['N_Stations_Dataset2'] = cruise_comparison['N_Stations_Dataset2'].fillna(0).astype(int)

# Keep Year and Month from whichever dataset has the cruise
cruise_comparison['Year'] = cruise_comparison['Year'].fillna(cruise_comparison['Year_2']).astype(int)
cruise_comparison['Month'] = cruise_comparison['Month'].fillna(cruise_comparison['Month_2']).astype(int)
cruise_comparison = cruise_comparison.drop(columns=['Year_2', 'Month_2'])

# Calculate total stations across both datasets
cruise_comparison['Total_Stations'] = (cruise_comparison['N_Stations_Dataset1'] + 
                                       cruise_comparison['N_Stations_Dataset2'])

# Calculate minimum (want cruises with good coverage in BOTH)
cruise_comparison['Min_Stations'] = cruise_comparison[['N_Stations_Dataset1', 
                                                       'N_Stations_Dataset2']].min(axis=1)

# ============================================================================
# FIND BEST CRUISES
# ============================================================================

print("\n" + "="*80)
print("TOP SPRING CRUISES BY TOTAL STATIONS")
print("="*80)

# Sort by total stations
best_by_total = cruise_comparison.sort_values('Total_Stations', ascending=False)

print("\nTop 20 cruises by TOTAL stations (Dataset1 + Dataset2):")
print(best_by_total.head(20).to_string())

print("\n" + "="*80)
print("TOP SPRING CRUISES BY MINIMUM STATIONS (Both datasets)")
print("="*80)

# Sort by minimum (ensures good coverage in BOTH)
best_by_min = cruise_comparison.sort_values('Min_Stations', ascending=False)

print("\nTop 20 cruises by MINIMUM stations (ensures coverage in both):")
print(best_by_min.head(20).to_string())

# ============================================================================
# SELECT BEST CRUISE
# ============================================================================

print("\n" + "="*80)
print("RECOMMENDED CRUISE SELECTION")
print("="*80)

# Strategy: Pick cruise with highest minimum (ensures both datasets have it)
# and high total (lots of stations overall)

# Filter to cruises that exist in BOTH datasets
both_datasets = cruise_comparison[
    (cruise_comparison['N_Stations_Dataset1'] > 0) & 
    (cruise_comparison['N_Stations_Dataset2'] > 0)
]

print(f"\nCruises in BOTH datasets: {len(both_datasets)}")

if len(both_datasets) > 0:
    # Sort by total stations
    best_cruise_id = both_datasets.sort_values('Total_Stations', ascending=False).index[0]
    
    print(f"\n✓ SELECTED CRUISE: {best_cruise_id}")
    print(f"\n  Year:  {both_datasets.loc[best_cruise_id, 'Year']}")
    print(f"  Month: {both_datasets.loc[best_cruise_id, 'Month']}")
    print(f"\n  Dataset 1 (predictions): {both_datasets.loc[best_cruise_id, 'N_Stations_Dataset1']} stations")
    print(f"  Dataset 2 (actual Secchi): {both_datasets.loc[best_cruise_id, 'N_Stations_Dataset2']} stations")
    print(f"  TOTAL: {both_datasets.loc[best_cruise_id, 'Total_Stations']} stations")
    
    # Save selected cruise ID
    with open("../Data/selected_spring_cruise.txt", 'w') as f:
        f.write(best_cruise_id)
    
    print(f"\n  Saved to: ../Data/selected_spring_cruise.txt")
    
else:
    print("\n⚠️  WARNING: No cruises found in BOTH datasets!")
    print("    Selecting from dataset with more spring cruises...")
    
    if len(cruise_counts_1) > len(cruise_counts_2):
        best_cruise_id = cruise_counts_1.sort_values('N_Stations_Dataset1', ascending=False).index[0]
        print(f"\n✓ SELECTED (from Dataset 1): {best_cruise_id}")
    else:
        best_cruise_id = cruise_counts_2.sort_values('N_Stations_Dataset2', ascending=False).index[0]
        print(f"\n✓ SELECTED (from Dataset 2): {best_cruise_id}")

# ============================================================================
# DETAILED INFO FOR SELECTED CRUISE
# ============================================================================

print("\n" + "="*80)
print("SELECTED CRUISE DETAILS")
print("="*80)

# Get stations from both datasets
selected_df1 = df1[df1['Cruise_ID'] == best_cruise_id]
selected_df2 = df2[df2['Cruise_ID'] == best_cruise_id]

print(f"\nCruise ID: {best_cruise_id}")

if len(selected_df1) > 0:
    print(f"\nDataset 1 (Predictions):")
    print(f"  Stations: {len(selected_df1)}")
    print(f"  Date range: {selected_df1['Date'].min()} to {selected_df1['Date'].max()}")
    print(f"  Lat range: {selected_df1['LAT_DEC'].min():.2f}° to {selected_df1['LAT_DEC'].max():.2f}°")
    print(f"  Lon range: {selected_df1['LON_DEC'].min():.2f}° to {selected_df1['LON_DEC'].max():.2f}°")
    print(f"  Predicted Secchi range: {selected_df1['Predicted_Secchi_RF'].min():.2f} to {selected_df1['Predicted_Secchi_RF'].max():.2f} m")
    print(f"  Predicted Secchi mean: {selected_df1['Predicted_Secchi_RF'].mean():.2f} m")
    print(f"  Predicted Secchi median: {selected_df1['Predicted_Secchi_RF'].median():.2f} m")
    print(f"  Predicted Secchi std: {selected_df1['Predicted_Secchi_RF'].std():.2f} m")

if len(selected_df2) > 0:
    print(f"\nDataset 2 (Actual Secchi):")
    print(f"  Stations: {len(selected_df2)}")
    print(f"  Date range: {selected_df2['Date'].min()} to {selected_df2['Date'].max()}")
    print(f"  Lat range: {selected_df2['Lat_Dec'].min():.2f}° to {selected_df2['Lat_Dec'].max():.2f}°")
    print(f"  Lon range: {selected_df2['Lon_Dec'].min():.2f}° to {selected_df2['Lon_Dec'].max():.2f}°")
    print(f"  Secchi range: {selected_df2['Secchi'].min():.2f} to {selected_df2['Secchi'].max():.2f} m")
    print(f"  Secchi mean: {selected_df2['Secchi'].mean():.2f} m")
    print(f"  Secchi median: {selected_df2['Secchi'].median():.2f} m")
    print(f"  Secchi std: {selected_df2['Secchi'].std():.2f} m")

# ============================================================================
# SAVE SUMMARY TABLE
# ============================================================================

# Save full comparison table
cruise_comparison.to_csv("../results/spring_cruise_comparison.csv")
print(f"\nFull comparison saved to: ../results/spring_cruise_comparison.csv")

print("\n" + "="*80)
print("CRUISE SELECTION COMPLETE!")
print("="*80)

Loading datasets...
Dataset 1 (predictions): 535 rows
Dataset 2 (actual Secchi): 1971 rows

Spring data:
  Dataset 1: 329 rows
  Dataset 2: 1047 rows

COUNTING STATIONS PER CRUISE

TOP SPRING CRUISES BY TOTAL STATIONS

Top 20 cruises by TOTAL stations (Dataset1 + Dataset2):
                   N_Stations_Dataset1  Year  Month  N_Stations_Dataset2  Total_Stations  Min_Stations
Cruise_ID                                                                                             
2006-07-08-C-32NM                   32  2006      7                   28              60            28
2005-04-15-C-32NM                   32  2005      4                   25              57            25
2005-07-01-C-32NM                   34  2005      7                   23              57            23
2018-04-05-C-325S                   20  2018      4                   26              46            20
2009-07-14-C-31M4                    9  2009      7                   34              43             9
2006

In [2]:
target_cruises = ['2020-10-11-C-33SR', '2019-07-11-C-32BH']

for cruise_id in target_cruises:
    print(f"{'='*60}")
    print(f"Cruise_ID: {cruise_id}")
    for label, df in [('Dataset 1 (predictions)', df1), ('Dataset 2 (actual Secchi)', df2)]:
        subset = df[df['Cruise_ID'] == cruise_id]
        if len(subset) == 0:
            print(f"  {label}: not found")
            continue
        n_rows = len(subset)
        n_unique_cast = subset['CAST_COUNT'].nunique()
        equivalent = n_rows == n_unique_cast
        print(f"  {label}:")
        print(f"    Rows (stations):      {n_rows}")
        print(f"    Unique CAST_COUNTs:   {n_unique_cast}")
        print(f"    Stations == Casts:    {equivalent}")
        if not equivalent:
            dupe_casts = subset[subset.duplicated('CAST_COUNT', keep=False)]['CAST_COUNT'].unique()
            print(f"    Duplicated CAST_COUNTs: {dupe_casts}")
print(f"{'='*60}")

Cruise_ID: 2020-10-11-C-33SR
  Dataset 1 (predictions):
    Rows (stations):      7
    Unique CAST_COUNTs:   7
    Stations == Casts:    True
  Dataset 2 (actual Secchi):
    Rows (stations):      20
    Unique CAST_COUNTs:   20
    Stations == Casts:    True
Cruise_ID: 2019-07-11-C-32BH
  Dataset 1 (predictions):
    Rows (stations):      9
    Unique CAST_COUNTs:   9
    Stations == Casts:    True
  Dataset 2 (actual Secchi):
    Rows (stations):      24
    Unique CAST_COUNTs:   24
    Stations == Casts:    True
